# Patch20 Clean Evaluation Notebook\n\nEvaluates an unmodified Patch20 zip in a fresh working directory. Any Kaggle compatibility overlay must be declared explicitly.

In [ ]:
from pathlib import Path\nimport json, os, shutil, subprocess, sys, zipfile\n\nPROJECT_ZIP = Path(os.getenv('PROJECT_ZIP', '/kaggle/input/cdss/cdss_professional_v4181_patch20_release_stabilized.zip'))\nWORKDIR = Path(os.getenv('WORKDIR', '/kaggle/working/cdss_patch20_clean'))\nOUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', '/kaggle/working/cdss_patch20_eval_outputs'))\nEVALUATION_OVERLAY_APPLIED = False\nOVERLAY_REASON = ''\nOVERLAY_FILES = []\n

In [ ]:
assert PROJECT_ZIP.exists(), f'Missing PROJECT_ZIP: {PROJECT_ZIP}'\nif WORKDIR.exists():\n    shutil.rmtree(WORKDIR)\nif OUTPUT_DIR.exists():\n    shutil.rmtree(OUTPUT_DIR)\nWORKDIR.mkdir(parents=True)\nOUTPUT_DIR.mkdir(parents=True)\nwith zipfile.ZipFile(PROJECT_ZIP) as zf:\n    zf.extractall(WORKDIR)\nprint({'project_zip': str(PROJECT_ZIP), 'workdir': str(WORKDIR), 'overlay_applied': EVALUATION_OVERLAY_APPLIED})\n

In [ ]:
env = os.environ.copy()\nenv['PYTHONPATH'] = str(WORKDIR) + os.pathsep + env.get('PYTHONPATH', '')\nenv['CDSS_FEEDBACK_DIR'] = str(OUTPUT_DIR / 'feedback')\nenv['FEEDBACK_DIR'] = str(OUTPUT_DIR / 'feedback')\nenv['AUDIT_DIR'] = str(OUTPUT_DIR / 'audit')\ncommands = [\n    [sys.executable, '-m', 'pytest', 'tests/unit/test_patch18_business_logic_core.py', 'tests/unit/test_patch19_monitoring_human_in_loop.py', 'tests/unit/test_patch20_release_regression.py', '-q'],\n]\nresults = []\nfor cmd in commands:\n    proc = subprocess.run(cmd, cwd=WORKDIR, env=env, text=True, capture_output=True)\n    results.append({'cmd': cmd, 'returncode': proc.returncode, 'stdout': proc.stdout[-4000:], 'stderr': proc.stderr[-4000:]})\n    print(results[-1])\n

In [ ]:
summary = {\n    'project_zip': str(PROJECT_ZIP),\n    'workdir': str(WORKDIR),\n    'overlay_applied': EVALUATION_OVERLAY_APPLIED,\n    'overlay_reason': OVERLAY_REASON,\n    'overlay_files': OVERLAY_FILES,\n    'results': results,\n}\n(OUTPUT_DIR / 'patch20_clean_eval_metrics.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')\n(OUTPUT_DIR / 'patch20_clean_eval_summary.md').write_text('Patch20 clean evaluation complete. See metrics JSON.\n', encoding='utf-8')\nsummary\n